In [ ]:
# === Full eval: held-out test (n=500, ROUGE/chrF/BLEU) + cultural (n=50, base+tuned) ===
# Loads public Qwen/Qwen2.5-1.5B-Instruct + public LoRA adapter, generates base vs tuned outputs
# greedy on both the embedded test slice and the inline cultural eval, computes auto-metrics,
# saves 4 JSON files matching the local src/eval_harness.py schemas so the local rubric/pairwise
# passes work against them without any further setup.
#
# DURABILITY: every partial+final save is also uploaded to HF Hub
# (tusharislampure29/qwen2.5-1.5b-marathi-instruct, eval_outputs/ subdir). Even if the Kaggle
# session is reaped mid-run, the latest partial files survive on HF.
#
# Outputs (under /kaggle/working/):
#   test_base.json, test_tuned.json          - schema: {model_id, adapter_id, tag, metrics, samples}
#   cultural_base.json, cultural_tuned.json  - schema: {model_id, adapter_id, tag, rubric, n, samples}
#   eval_outputs.zip                         - bundle of all four for one-click download

import base64, gzip, json, os, subprocess, time
from collections import defaultdict

# 0. HF_TOKEN for durable uploads (substituted from local env at build time)
HF_TOKEN = "__HF_TOKEN_PLACEHOLDER__"
HF_REPO  = "tusharislampure29/qwen2.5-1.5b-marathi-instruct"
HF_SUBDIR = "eval_outputs"

# 1. install
for pkg in ["transformers", "peft", "bitsandbytes", "accelerate", "rouge_score", "sacrebleu", "huggingface_hub"]:
    subprocess.run(["pip", "install", "-q", "-U", pkg], capture_output=True)

from huggingface_hub import HfApi
_hf_api = HfApi(token=HF_TOKEN) if HF_TOKEN and not HF_TOKEN.startswith("__") else None
def hf_push(local_path, repo_path):
    if _hf_api is None:
        print(f"  [hf_push] skipped (no token): {local_path}")
        return
    try:
        _hf_api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=f"{HF_SUBDIR}/{repo_path}",
            repo_id=HF_REPO, repo_type="model",
            commit_message=f"eval: {repo_path}",
        )
        print(f"  [hf_push] OK: {repo_path}")
    except Exception as e:
        print(f"  [hf_push] FAIL {repo_path}: {e}")

import torch
torch.manual_seed(42)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

CFG = {
    "base_id":          "Qwen/Qwen2.5-1.5B-Instruct",
    "adapter_id":       "tusharislampure29/qwen2.5-1.5b-marathi-instruct",
    "system_prompt":    "तुम्ही एक उपयुक्त AI सहाय्यक आहात जो मराठीत स्पष्ट आणि अचूक उत्तरे देतो.",
    "max_new_cultural": 200,
    "max_new_test":     256,
    "out_dir":          "/kaggle/working",
}

# 2. inline test slice (first 500 rows of held-out test.jsonl, gzip+b64)
TEST_B64 = "..."
TEST_ROWS = json.loads(gzip.decompress(base64.b64decode(TEST_B64)).decode("utf-8"))
print(f"test rows embedded: {len(TEST_ROWS)}")

# 3. inline cultural eval (50 prompts)
CULTURAL = json.loads(r"""{"version":"1.0","language":"marathi","created":"2026-05-11","author":"Tushar Islampure","purpose":"Hand-curated Marathi evaluation set for measuring fluency, cultural accuracy, factuality, and reasoning of Marathi-tuned LLMs. Used with src/eval_harness.py --rubric openai. Built because off-the-shelf BLEU/ROUGE on translated benchmarks misses what Marathi speakers actually care about.","categories":["geography","history","culture","language","reasoning"],"system_prompt_marathi":"तुम्ही एक उपयुक्त AI सहाय्यक आहात जो मराठीत स्पष्ट आणि अचूक उत्तरे देतो.","prompts":[{"id":"geo-01","category":"geography","difficulty":"easy","prompt":"महाराष्ट्राची राजधानी कोणती आहे?","expected_keywords":["मुंबई"],"notes":"Capital of Maharashtra. Single-token answer."},{"id":"geo-02","category":"geography","difficulty":"easy","prompt":"महाराष्ट्रातील दोन सर्वात मोठी शहरे कोणती?","expected_keywords":["मुंबई","पुणे"],"notes":"Two largest cities."},{"id":"geo-03","category":"geography","difficulty":"medium","prompt":"महाराष्ट्रातील पश्चिमेकडे असलेल्या डोंगररांगेला कोणते नाव आहे?","expected_keywords":["सह्याद्री","पश्चिम घाट"],"notes":"Sahyadri / Western Ghats."},{"id":"geo-04","category":"geography","difficulty":"medium","prompt":"महाराष्ट्रातून वाहणारी एक मोठी नदी कोणती?","expected_keywords":["गोदावरी","कृष्णा","भीमा","तापी"],"notes":"Any major river: Godavari, Krishna, Bhima, Tapi."},{"id":"geo-05","category":"geography","difficulty":"easy","prompt":"महाराष्ट्राच्या पश्चिम किनाऱ्याला कोणता समुद्र आहे?","expected_keywords":["अरबी"],"notes":"Arabian Sea."},{"id":"geo-06","category":"geography","difficulty":"medium","prompt":"विदर्भ प्रांतातील दोन प्रमुख शहरे कोणती?","expected_keywords":["नागपूर","अमरावती"],"notes":"Nagpur, Amravati (and Akola, Wardha also acceptable)."},{"id":"geo-07","category":"geography","difficulty":"medium","prompt":"कोकण प्रदेश महाराष्ट्राच्या कोणत्या भागात आहे?","expected_keywords":["पश्चिम","किनारा"],"notes":"West coast (Konkan)."},{"id":"geo-08","category":"geography","difficulty":"hard","prompt":"महाराष्ट्रात किती जिल्हे आहेत?","expected_keywords":["३६","36","छत्तीस"],"notes":"36 districts as of recent times. Will accept ±2."},{"id":"geo-09","category":"geography","difficulty":"easy","prompt":"नाशिक हे शहर कोणत्या नदीच्या काठावर वसले आहे?","expected_keywords":["गोदावरी"],"notes":"Nashik is on the Godavari river."},{"id":"geo-10","category":"geography","difficulty":"medium","prompt":"महाराष्ट्राच्या शेजारी असलेली तीन राज्ये सांगा.","expected_keywords":["गुजरात","मध्य प्रदेश","तेलंगणा","कर्नाटक","गोवा","छत्तीसगड"],"notes":"Any three of Maharashtra's neighbours."},{"id":"hist-01","category":"history","difficulty":"easy","prompt":"मराठा साम्राज्याचे संस्थापक कोण होते?","expected_keywords":["शिवाजी"],"notes":"Chhatrapati Shivaji Maharaj."},{"id":"hist-02","category":"history","difficulty":"medium","prompt":"छत्रपती शिवाजी महाराजांचा राज्याभिषेक कोणत्या किल्ल्यावर झाला?","expected_keywords":["रायगड"],"notes":"Raigad Fort."},{"id":"hist-03","category":"history","difficulty":"medium","prompt":"'गुलामगिरी' हे प्रसिद्ध पुस्तक कोणी लिहिले?","expected_keywords":["ज्योतिबा फुले","जोतीराव फुले","महात्मा फुले"],"notes":"Jyotirao (Mahatma) Phule wrote Gulamgiri."},{"id":"hist-04","category":"history","difficulty":"easy","prompt":"'लोकमान्य' ही पदवी कोणाला दिली होती?","expected_keywords":["टिळक","बाळ गंगाधर टिळक"],"notes":"Bal Gangadhar Tilak."},{"id":"hist-05","category":"history","difficulty":"medium","prompt":"पहिल्या स्त्री शिक्षिकेचे नाव सांगा ज्यांनी पुण्यात मुलींची शाळा सुरू केली.","expected_keywords":["सावित्रीबाई फुले"],"notes":"Savitribai Phule — opened first girls' school in Pune."},{"id":"hist-06","category":"history","difficulty":"easy","prompt":"डॉ. बाबासाहेब आंबेडकरांनी कोणत्या समाजाच्या उद्धारासाठी कार्य केले?","expected_keywords":["दलित","अस्पृश्य"],"notes":"Dalit upliftment."},{"id":"hist-07","category":"history","difficulty":"hard","prompt":"अफजल खान आणि शिवाजी महाराजांची भेट कोणत्या किल्ल्यावर झाली होती?","expected_keywords":["प्रतापगड"],"notes":"Pratapgad — site of Afzal Khan's meeting/killing."},{"id":"hist-08","category":"history","difficulty":"medium","prompt":"संभाजी महाराज हे कोणाचे पुत्र होते?","expected_keywords":["शिवाजी"],"notes":"Sambhaji was son of Shivaji."},{"id":"hist-09","category":"history","difficulty":"hard","prompt":"मराठा साम्राज्याचे शेवटचे पेशवे कोण होते?","expected_keywords":["बाजीराव","दुसरा बाजीराव","बाजीराव दुसरा"],"notes":"Bajirao II — last Peshwa."},{"id":"hist-10","category":"history","difficulty":"medium","prompt":"महात्मा गांधींबद्दल तीन वाक्यांत माहिती द्या.","expected_keywords":["स्वातंत्र्य","अहिंसा","गांधी"],"notes":"Open-ended. Judge on accuracy + 3-sentence constraint."},{"id":"cult-01","category":"culture","difficulty":"easy","prompt":"गणपती बाप्पाला सर्वात आवडता गोड पदार्थ कोणता?","expected_keywords":["मोदक"],"notes":"Modak is Ganesha's favourite sweet."},{"id":"cult-02","category":"culture","difficulty":"easy","prompt":"गुढीपाडवा कोणत्या महिन्यात येतो?","expected_keywords":["चैत्र","मार्च","एप्रिल"],"notes":"Chaitra Shudh Pratipada — March/April. Marathi New Year."},{"id":"cult-03","category":"culture","difficulty":"medium","prompt":"'पुरणपोळी' म्हणजे काय?","expected_keywords":["पुरण","पोळी","गोड"],"notes":"Sweet flatbread stuffed with gram/jaggery purna."},{"id":"cult-04","category":"culture","difficulty":"medium","prompt":"महाराष्ट्रातील प्रसिद्ध लोकनृत्य कोणते?","expected_keywords":["लावणी"],"notes":"Lavani."},{"id":"cult-05","category":"culture","difficulty":"medium","prompt":"'वारकरी' म्हणजे कोण?","expected_keywords":["विठ्ठल","पंढरपूर","वारी"],"notes":"Warkari — devotees of Vitthal who walk to Pandharpur."},{"id":"cult-06","category":"culture","difficulty":"medium","prompt":"संत तुकाराम कोणत्या गावाशी संबंधित आहेत?","expected_keywords":["देहू"],"notes":"Dehu."},{"id":"cult-07","category":"culture","difficulty":"medium","prompt":"संत ज्ञानेश्वर कोणत्या गावात समाधीस्थ झाले?","expected_keywords":["आळंदी"],"notes":"Alandi."},{"id":"cult-08","category":"culture","difficulty":"easy","prompt":"पंढरपूरमध्ये कोणत्या देवाचे प्रसिद्ध मंदिर आहे?","expected_keywords":["विठ्ठल","विठोबा"],"notes":"Vitthal / Vithoba."},{"id":"cult-09","category":"culture","difficulty":"medium","prompt":"महाराष्ट्रात सार्वजनिक गणेशोत्सव सुरू करण्याचे श्रेय कोणाला जाते?","expected_keywords":["टिळक","लोकमान्य टिळक"],"notes":"Lokmanya Tilak."},{"id":"cult-10","category":"culture","difficulty":"medium","prompt":"महाराष्ट्रातील एक प्रसिद्ध शाकाहारी जेवण सांगा.","expected_keywords":["पुरणपोळी","वरण","भात","भाजी","थालिपीठ","मिसळ"],"notes":"Any vegetarian dish. Multiple valid."},{"id":"lang-01","category":"language","difficulty":"easy","prompt":"'पुस्तक' या शब्दाला इंग्रजीत काय म्हणतात?","expected_keywords":["book"],"notes":"Book."},{"id":"lang-02","category":"language","difficulty":"easy","prompt":"मराठी मधे 'computer' ला काय म्हणतात?","expected_keywords":["संगणक"],"notes":"Sanganak."},{"id":"lang-03","category":"language","difficulty":"medium","prompt":"'आगीतून फुफाट्यात' या वाक्प्रचाराचा अर्थ काय?","expected_keywords":["संकट","मोठ्या","दुसऱ्या"],"notes":"Out of frying pan into the fire — from one bigger trouble to another."},{"id":"lang-04","category":"language","difficulty":"medium","prompt":"'गाढवापुढे वाचली गीता' या म्हणीचा अर्थ काय?","expected_keywords":["समजत","मूर्ख","व्यर्थ","उपयोग"],"notes":"Reading Gita to a donkey — wasted advice on a foolish person."},{"id":"lang-05","category":"language","difficulty":"easy","prompt":"'उद्या' या शब्दाचा अर्थ काय? एक वाक्यात सांगा.","expected_keywords":["येणारा","दिवस","पुढचा"],"notes":"Tomorrow / next day."},{"id":"lang-06","category":"language","difficulty":"medium","prompt":"'अंथरूण पाहून पाय पसरावे' या म्हणीचा अर्थ काय?","expected_keywords":["मर्यादा","ऐपत","आर्थिक","खर्च"],"notes":"Spread your legs according to your bed — live within means."},{"id":"lang-07","category":"language","difficulty":"medium","prompt":"'मोडकं तोडकं' म्हणजे काय?","expected_keywords":["अपूर्ण","थोडे","अस्खलित नाही"],"notes":"Broken / partial (e.g., broken Marathi)."},{"id":"lang-08","category":"language","difficulty":"hard","prompt":"'यायाती' ही प्रसिद्ध मराठी कादंबरी कोणी लिहिली?","expected_keywords":["खांडेकर","वि. स. खांडेकर"],"notes":"V.S. Khandekar — Yayati won the Jnanpith."},{"id":"lang-09","category":"language","difficulty":"medium","prompt":"'पु. ल. देशपांडे' हे कोण होते?","expected_keywords":["लेखक","विनोदी","मराठी","साहित्यिक"],"notes":"Famous Marathi humorist / author."},{"id":"lang-10","category":"language","difficulty":"easy","prompt":"'I am going home' या इंग्रजी वाक्याचे मराठी भाषांतर सांगा.","expected_keywords":["मी","घरी","जात","जातो","जाते"],"notes":"Mi ghari jato/jate (depending on gender)."},{"id":"reas-01","category":"reasoning","difficulty":"easy","prompt":"२५ + ७५ किती होतात?","expected_keywords":["१००","100","शंभर"],"notes":"Simple arithmetic."},{"id":"reas-02","category":"reasoning","difficulty":"medium","prompt":"रामची उंची सीतापेक्षा जास्त आहे, आणि सीताची उंची मीनापेक्षा जास्त आहे. तर सर्वात कमी उंचीची व्यक्ती कोण?","expected_keywords":["मीना"],"notes":"Transitive reasoning."},{"id":"reas-03","category":"reasoning","difficulty":"medium","prompt":"जर एक रेल्वे संध्याकाळी ६ वाजता मुंबईहून निघाली आणि रात्री ९:३० वाजता पुण्यात पोहोचली, तर प्रवास किती वेळ चालला?","expected_keywords":["३ तास ३०","साडे तीन","३.५","3.5","210 मिनिटे"],"notes":"3 hours 30 minutes."},{"id":"reas-04","category":"reasoning","difficulty":"easy","prompt":"उन्हात बर्फ का वितळतो?","expected_keywords":["उष्णता","तापमान","सूर्य"],"notes":"Because of sun's heat."},{"id":"reas-05","category":"reasoning","difficulty":"medium","prompt":"एका दुकानात एक पेन ३० रुपयांना मिळतो. १२ पेन घेतल्यास किती रुपये द्यावे लागतील?","expected_keywords":["३६०","360"],"notes":"30 * 12 = 360."},{"id":"reas-06","category":"reasoning","difficulty":"easy","prompt":"एका आठवड्यात किती दिवस असतात?","expected_keywords":["सात","७","7"],"notes":"Seven days in a week."},{"id":"reas-07","category":"reasoning","difficulty":"medium","prompt":"एक चांगला सॉफ्टवेअर इंजिनिअर होण्यासाठी कोणते तीन गुण आवश्यक आहेत?","expected_keywords":["समस्या","शिकण्याची","तर्क","संवाद","अभ्यास"],"notes":"Open-ended. Judge on relevance + coherence."},{"id":"reas-08","category":"reasoning","difficulty":"medium","prompt":"मुलांना दररोज व्यायाम करण्याचे तीन फायदे सांगा.","expected_keywords":["आरोग्य","एकाग्रता","ताकद","तंदुरुस्ती"],"notes":"Open-ended. Judge on accuracy."},{"id":"reas-09","category":"reasoning","difficulty":"hard","prompt":"जर आज रविवार असेल तर १० दिवसांनी कोणता वार येईल?","expected_keywords":["बुधवार"],"notes":"Sun + 10 = Wed."},{"id":"reas-10","category":"reasoning","difficulty":"medium","prompt":"स्वच्छ पाणी पिण्याचे महत्त्व दोन वाक्यांत सांगा.","expected_keywords":["आरोग्य","रोग","शुद्ध","जंतू"],"notes":"Open-ended. Judge clarity + 2-sentence constraint."}]}""")
print(f"cultural prompts: {len(CULTURAL['prompts'])}")

# 4. load base + attach LoRA adapter (both public, no token needed)
# GPU-aware: bnb 4-bit needs Turing or newer (T4 = 7.5, P100 = 6.0). On Pascal we fall back to fp16,
# which still fits the 1.5B model with plenty of headroom in P100's 16 GB.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
_compute_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
_use_4bit = _compute_cap[0] >= 7  # Turing (T4) is 7.5, Ampere/Hopper are higher
print(f"GPU: {_gpu_name}  compute_capability: {_compute_cap}  use_4bit: {_use_4bit}")

tok = AutoTokenizer.from_pretrained(CFG["base_id"])
if _use_4bit:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    _base = AutoModelForCausalLM.from_pretrained(CFG["base_id"], quantization_config=bnb,
                                                  device_map="auto", torch_dtype=torch.float16)
else:
    _base = AutoModelForCausalLM.from_pretrained(CFG["base_id"], device_map="auto",
                                                  torch_dtype=torch.float16)
model = PeftModel.from_pretrained(_base, CFG["adapter_id"])
model.eval()
print("model + adapter loaded")

# 5. greedy generate (one prompt, one mode)
def gen(prompt: str, max_new: int, use_adapter: bool) -> str:
    msgs = [{"role": "system", "content": CFG["system_prompt"]},
            {"role": "user",   "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new, do_sample=False, temperature=1.0, top_p=1.0,
                  pad_token_id=tok.eos_token_id)
    if use_adapter:
        with torch.no_grad():
            out = model.generate(**inputs, **kwargs)
    else:
        with model.disable_adapter(), torch.no_grad():
            out = model.generate(**inputs, **kwargs)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def write_json(path: str, payload: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"  wrote {path} ({os.path.getsize(path)/1024:.0f} KB)")

# 6. cultural first (faster - if test crashes we still have this)
from tqdm.auto import tqdm
print("\n--- cultural generation (n=50 x 2) ---")
t0 = time.time()
base_cult, tuned_cult = [], []
for item in tqdm(CULTURAL["prompts"]):
    common = {"id": item["id"], "category": item["category"], "difficulty": item["difficulty"],
              "prompt": item["prompt"], "expected_keywords": item.get("expected_keywords", [])}
    base_cult.append({**common, "response": gen(item["prompt"], CFG["max_new_cultural"], False)})
    tuned_cult.append({**common, "response": gen(item["prompt"], CFG["max_new_cultural"], True)})
print(f"cultural done in {(time.time()-t0)/60:.1f} min")

write_json(f"{CFG['out_dir']}/cultural_base.json", {
    "model_id": CFG["base_id"], "adapter_id": None, "tag": "cultural_base",
    "rubric": None, "n": len(base_cult), "samples": base_cult})
write_json(f"{CFG['out_dir']}/cultural_tuned.json", {
    "model_id": CFG["base_id"], "adapter_id": CFG["adapter_id"], "tag": "cultural_tuned",
    "rubric": None, "n": len(tuned_cult), "samples": tuned_cult})
hf_push(f"{CFG['out_dir']}/cultural_base.json",  "cultural_base.json")
hf_push(f"{CFG['out_dir']}/cultural_tuned.json", "cultural_tuned.json")

# 7. test set (long pass - ~80 min for n=500)
print(f"\n--- test generation (n={len(TEST_ROWS)} x 2) ---")
t0 = time.time()
base_test, tuned_test = [], []
for row in tqdm(TEST_ROWS):
    instr, ref = row["instruction"], row["response"]
    b = gen(instr, CFG["max_new_test"], False)
    t = gen(instr, CFG["max_new_test"], True)
    base_test.append({"instruction": instr, "reference": ref, "prediction": b})
    tuned_test.append({"instruction": instr, "reference": ref, "prediction": t})
    # incremental save every 100 in case of OOM/timeout + durable push to HF Hub
    if (len(base_test) % 100) == 0:
        write_json(f"{CFG['out_dir']}/test_base.partial.json", {
            "model_id": CFG["base_id"], "adapter_id": None, "tag": "test_base_partial",
            "metrics": {"n": len(base_test)}, "samples": base_test})
        write_json(f"{CFG['out_dir']}/test_tuned.partial.json", {
            "model_id": CFG["base_id"], "adapter_id": CFG["adapter_id"], "tag": "test_tuned_partial",
            "metrics": {"n": len(tuned_test)}, "samples": tuned_test})
        hf_push(f"{CFG['out_dir']}/test_base.partial.json",  f"test_base.partial_{len(base_test):04d}.json")
        hf_push(f"{CFG['out_dir']}/test_tuned.partial.json", f"test_tuned.partial_{len(tuned_test):04d}.json")
print(f"test done in {(time.time()-t0)/60:.1f} min")

# 8. auto-metrics
from rouge_score import rouge_scorer
import sacrebleu
def compute_metrics(preds, refs):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rouge_l = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
    return {"rougeL_f1": (sum(rouge_l)/len(rouge_l)) if rouge_l else 0.0,
            "chrF": sacrebleu.corpus_chrf(preds, [refs]).score,
            "bleu": sacrebleu.corpus_bleu(preds, [refs]).score,
            "n":    len(preds)}

refs    = [s["reference"]  for s in base_test]
m_base  = compute_metrics([s["prediction"] for s in base_test],  refs)
m_tuned = compute_metrics([s["prediction"] for s in tuned_test], refs)

write_json(f"{CFG['out_dir']}/test_base.json", {
    "model_id": CFG["base_id"], "adapter_id": None, "tag": "test_base",
    "metrics": m_base, "samples": base_test})
write_json(f"{CFG['out_dir']}/test_tuned.json", {
    "model_id": CFG["base_id"], "adapter_id": CFG["adapter_id"], "tag": "test_tuned",
    "metrics": m_tuned, "samples": tuned_test})
hf_push(f"{CFG['out_dir']}/test_base.json",  "test_base.json")
hf_push(f"{CFG['out_dir']}/test_tuned.json", "test_tuned.json")

# 9. keyword recall on cultural (sanity check)
def kw_recall_by_cat(rows):
    by_cat, overall = defaultdict(list), []
    for r in rows:
        kws = r.get("expected_keywords") or []
        if not kws: continue
        hits = sum(1 for k in kws if k in r["response"]) / len(kws)
        by_cat[r["category"]].append(hits); overall.append(hits)
    return {"overall": (sum(overall)/len(overall)) if overall else 0.0,
            "by_category": {c: sum(v)/len(v) for c, v in by_cat.items()}}
kb = kw_recall_by_cat(base_cult); kt = kw_recall_by_cat(tuned_cult)

# 10. bundle
import zipfile
zip_path = f"{CFG['out_dir']}/eval_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ["test_base.json", "test_tuned.json", "cultural_base.json", "cultural_tuned.json"]:
        z.write(f"{CFG['out_dir']}/{fn}", arcname=fn)
print(f"\nbundle: {zip_path} ({os.path.getsize(zip_path)/1024:.0f} KB)")
hf_push(zip_path, "eval_outputs.zip")

# 11. headline
print("\n=== HEADLINE ===")
print(f"test set (n={m_base['n']}):")
print(f"  rougeL_f1: base={m_base['rougeL_f1']:.4f}  tuned={m_tuned['rougeL_f1']:.4f}  delta={m_tuned['rougeL_f1']-m_base['rougeL_f1']:+.4f}")
print(f"  chrF     : base={m_base['chrF']:6.2f}   tuned={m_tuned['chrF']:6.2f}   delta={m_tuned['chrF']-m_base['chrF']:+.2f}")
print(f"  bleu     : base={m_base['bleu']:6.2f}   tuned={m_tuned['bleu']:6.2f}   delta={m_tuned['bleu']-m_base['bleu']:+.2f}")
print(f"\ncultural keyword recall (n=50):")
print(f"  overall: base={kb['overall']:.3f}  tuned={kt['overall']:.3f}  delta={kt['overall']-kb['overall']:+.3f}")
for c in kt["by_category"]:
    b, t = kb["by_category"][c], kt["by_category"][c]
    print(f"  {c:12s}: base={b:.3f}  tuned={t:.3f}  delta={t-b:+.3f}")

print("\n=== DONE ===")
